## Feature Implementation Demos

This notebook demonstrates the **project features** using the same **unified interface** as `evaluation_basic.py`:
- `rag = RAGEngine(...)`
- `pm = PromptManager()`
- `chat = ChatLogic(rag, pm)`
- `chat.process_query(query, retrieval_type=..., top_k=...)` → returns `response`, `cited_docs`, `total_tokens`, etc.

This notebook focuses on:
- **Multi-turn conversation memory** (via `ChatLogic.history`)
- **Study-plan templates** (input/output)
- **Study-plan demo** using **Neural RAG** (prints `response`, `cited_docs`, and `total_tokens`)


In [5]:
import os, sys

PROJECT_ROOT = r"e:\HKBU\COMP7125 Prompt Engineering\lab\HKBU_Study_Companion-main\HKBU_Study_Companion-main"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

# Align with the basic unified interface
from src.rag_engine import RAGEngine
from src.prompt_manager import PromptManager
from src.chat_logic import ChatLogic

rag = RAGEngine(chunk_file="chunks_natural_500_50.jsonl")
pm = PromptManager()
chat = ChatLogic(rag, pm)

# ChatLogic uses ollama.generate(model="gemma3:4b") internally.
GEN_MODEL = "gemma3:4b"



cwd: e:\HKBU\COMP7125 Prompt Engineering\lab\HKBU_Study_Companion-main\HKBU_Study_Companion-main
 Initializing RAG Engine with chunks_natural_500_50.jsonl
Loaded 401 chunks from data\chunks_natural_500_50.jsonl
No cache found → generating embeddings (this runs only once)...
   Processed 50/401
   Processed 100/401
   Processed 150/401
   Processed 200/401
   Processed 250/401
   Processed 300/401
   Processed 350/401
   Processed 400/401
Embeddings cached to vector_db\chunks_natural_500_50_embeddings.npy
Initializing Lexical Retriever...
Initializing Neural Retriever (using cache)...
RAG Engine ready! (401 chunks)


### 1) Multi-turn conversation memory

This demo uses the **unified interface** (`chat.process_query`) twice in a row.

- The second user turn is evaluated with **conversation history already stored** in `chat.history`.
- The output includes **token usage** (`total_tokens`) and **citations** (`cited_docs`).

In [6]:
# Multi-turn memory using the unified interface (ChatLogic.history)
chat.clear_history()

user_1 = "I have 10 hours per week. Make me a 2-week study plan for COMP7300."
res_1 = chat.process_query(user_1, retrieval_type="neural", top_k=3)

user_2 = "Update the plan: I can only study on weekends, but I want to keep the same total hours."
res_2 = chat.process_query(user_2, retrieval_type="neural", top_k=3)

print("User 1:\n", user_1)
print("\nAssistant 1:\n", res_1["response"])
print("\nTokens 1:", res_1["total_tokens"], "| Cited:", res_1["cited_docs"][:2], "...")

print("\nUser 2:\n", user_2)
print("\nAssistant 2:\n", res_2["response"])
print("\nTokens 2:", res_2["total_tokens"], "| Cited:", res_2["cited_docs"][:2], "...")


User 1:
 I have 10 hours per week. Make me a 2-week study plan for COMP7300.

Assistant 1:
 I understand you're looking for a 2-week study plan for COMP7300. However, the provided context doesn’t contain information about COMP7300. It lists courses like COMP7015, COMP7180, COMP7950, COMP7990, COMP7035, and COMP7530. 

Could you please confirm the course code for which you need a study plan? Or, if you’re interested in one of the courses listed, please specify which one (e.g., COMP7015 Artificial Intelligence)?

Tokens 1: 891 | Cited: ['D:\\Study lecture\\7125 Prompt\\HKBU_Study_Companion-main\\data\\Programme Handout for DAAI.pdf', 'D:\\Study lecture\\7125 Prompt\\HKBU_Study_Companion-main\\data\\Programme Handout for DAAI.pdf'] ...

User 2:
 Update the plan: I can only study on weekends, but I want to keep the same total hours.

Assistant 2:
 I understand you’d like to adjust your study plan to focus on weekends while maintaining the 10 hours per week. However, the provided context do

### 2) Study-plan suggestion templates (input/output)

These templates match how the assistant is prompted by `PromptManager` and what we expect `chat.process_query(...)` to produce.

**Input template (user constraints):**
- Courses
- Exam dates / deadlines (if known)
- Available study hours (weekday/weekend)
- Goals (target grade / focus topics)
- Preferences (e.g., weekends only / evenings only)

**Output template (assistant):**
- Assumptions / missing info
- Day-by-day or week-by-week schedule
- Course/topic breakdown
- Risk flags (tight schedule / missing exam dates)
- Citations (e.g., `[1]`) + a short list of sources (`cited_docs`)

In [7]:
study_query = (
    "Today's date is 2026-04-08. Create a revision plan for my finals for COMP7880, COMP7870, and COMP7300. "
    "Use the exam timetable and any course policy information from the documents. "
    "I can study 2 hours on weekdays and 5 hours on weekends. Provide a day-by-day plan."
)

chat.clear_history()
res = chat.process_query(study_query, retrieval_type="neural", top_k=5)

print("Tokens:", res["total_tokens"])
print("Cited (first 3):", res["cited_docs"][:3])
print("\nPlan:\n", res["response"])


Tokens: 1879
Cited (first 3): ['D:\\Study lecture\\7125 Prompt\\HKBU_Study_Companion-main\\data\\Examination Timetable.pdf', 'D:\\Study lecture\\7125 Prompt\\HKBU_Study_Companion-main\\data\\Examination Timetable2.pdf', 'D:\\Study lecture\\7125 Prompt\\HKBU_Study_Companion-main\\data\\Examination Timetable2.pdf']

Plan:
 Okay, let’s create a revision plan for your finals, considering the available information.

**Course Details:**

*   **COMP7880 Special Topics in Knowledge & Info Mgnt:** Examination date is 09-05-2026 (Sat) from 13:00 - 16:00 [2]
*   **COMP7870:**  *No specific examination details are listed in the provided timetable.* Please consult the course instructor for assessment methods.
*   **COMP7300:** *No specific examination details are listed in the provided timetable.* Please consult the course instructor for assessment methods.

**Revision Plan (Based on Timetable & General Information):**

**Important Note:** The timetable indicates that individual examination schedul